In [ ]:
import pandas as pd
import os
import re

In [ ]:

file_names = os.listdir('./data/전처리_완료')
file_names

In [ ]:
import re
from konlpy.tag import Okt

okt = Okt()

def clean_text(text):
    # 특수문자 제거, 영어/숫자 선택적 포함 가능
    text = re.sub(r'[^가-힣\s]', ' ', text)  # 한글과 공백만 남기기
    text = re.sub(r'\s+', ' ', text).strip()  # 공백 정리
    return text

def extract_keywords_with_negation(text):
    if not text or not isinstance(text, str):
        return []

    text = re.sub(r'[^\w\s가-힣]', '', text)
    morphs = okt.pos(text, norm=True, stem=True)
    result = []
    i = 0

    while i < len(morphs):
        word, tag = morphs[i]

        # Case 1: '지 않다' or '지 못하다'
        if word == '지' and i + 1 < len(morphs):
            next_word, next_tag = morphs[i + 1]
            if next_word in ['않다', '못하다'] and next_tag == 'Verb':
                if result:
                    prev = result.pop()
                    result.append(f'NOT_{prev}')
                i += 2
                continue

        # Case 2: '않다', '못하다', '없다' (보조용언/형용사 → 앞 단어 부정)
        if word in ['않다', '못하다', '없다'] and tag in ['Verb', 'Adjective']:
            if result:
                prev = result.pop()
                result.append(f'NOT_{prev}')
            i += 1
            continue

        # ✅ Case 3: 부정 부사 (안, 못, 아니)
        if word in ['안', '못', '아니'] and tag in ['Adverb', 'Noun']:
            if i + 1 < len(morphs):
                next_word, next_tag = morphs[i + 1]

                if next_tag in ['Verb', 'Adjective']:
                    # 👇 수정된 부분: '나다/보이다/들리다'는 앞 명사 부정
                    if result and result[-1] and next_word in ['나다', '보이다', '들리다']:
                        prev = result[-1]  # pop하지 않고 유지
                        result.append(f'NOT_{next_word}')  # 동사에 부정
                    else:
                        result.append(f'NOT_{next_word}')
                    i += 2
                    continue


        # 일반 키워드: 명사/동사/형용사
        if tag in ['Noun', 'Verb', 'Adjective']:
            result.append(word)

        i += 1

    return result

In [ ]:
for i in file_names:
    print(i)
    df = pd.read_csv('./data/전처리_완료/' + i)
    df['token'] = df['리뷰'].apply(clean_text).apply(extract_keywords_with_negation)
    df.to_csv('./data/전처리_완료/' + i, index=False)

In [ ]:
df_dggb_1 = pd.read_csv('./data/전처리_완료/구분O_건봉국밥_리뷰_전처리.csv')
df_dggb_2 = pd.read_csv('./data/전처리_완료/구분O_꼰대국밥_리뷰_전처리.csv')
df_dggb_3 = pd.read_csv('./data/전처리_완료/구분O_대건명가_리뷰_전처리.csv')
df_dggb_4 = pd.read_csv('./data/전처리_완료/구분O_몽실종가_리뷰_전처리.csv')
df_dggb_5 = pd.read_csv('./data/전처리_완료/구분O_수백당_돼지국밥_리뷰_전처리.csv')
df_dggb_6 = pd.read_csv('./data/전처리_완료/구분O_장사의신_돼지국밥_리뷰_전처리.csv')

In [ ]:
def remove_duplicated_token(tokens):
    seen = set()
    result = []
    for token in tokens:
        if token not in seen:
            seen.add(token)
            result.append(token)
    return result

In [ ]:
import ast

def clean_dataframe(df):
    df['token'] = df['token'].apply(ast.literal_eval)
    df['token'] = df['token'].apply(remove_duplicated_token)
    return df.drop_duplicates(subset='리뷰', keep='first')

df_dggb_1 = clean_dataframe(df_dggb_1)
df_dggb_2 = clean_dataframe(df_dggb_2)
df_dggb_3 = clean_dataframe(df_dggb_3)
df_dggb_4 = clean_dataframe(df_dggb_4)
df_dggb_5 = clean_dataframe(df_dggb_5)
df_dggb_6 = clean_dataframe(df_dggb_6)

In [ ]:
from itertools import chain

all_tokens = list(chain.from_iterable(
    df['token'] for df in [df_dggb_1, df_dggb_2, df_dggb_3, df_dggb_4, df_dggb_5, df_dggb_6]  # 또는 'tokens_dedup'
))

In [ ]:
# 빈도수 계산
from collections import Counter
token_freq = Counter(chain.from_iterable(all_tokens))

# DataFrame으로 보기 좋게 변환
freq_df = pd.DataFrame(token_freq.items(), columns=['토큰', '빈도수']).sort_values(by='빈도수', ascending=False)
freq_df

In [ ]:
freq_df[freq_df['토큰'].str.contains('NOT')]

In [ ]:
freq_df.to_excel('./data/토큰_빈도수.xlsx', index=False)

In [ ]:
from collections import Counter
from itertools import chain
import numpy as np
import pandas as pd

# 1. 맛 관련 키워드 리스트 (개별 단어)
taste_keywords = [
    '깔끔하다', '넉넉하다', '진하다',
    '든든하다', '담백하다', '부드럽다',
    '구수하다', '실하다', '깊다',
    '짜다', '짭잘하다', '짭짤하다',
    '맵다', '얼큰하다', '고소하다',
    '느끼하다', '시원하다', '진국',
    '뽀얗다', '싱겁다', '매콤',
    '퍽퍽', '칼칼하다', '맑다',
    '자극', '감칠맛', '맑은',
    '개운하다', '야들야들', '야들야들하다',
    '쫄깃', '쫄깃쫄깃', '걸쭉하다', '묽다'
]

# 2. 맛 그룹 딕셔너리 (필요한 단어만 그룹화, 예: '맑다'에 '맑다', '맑은' 묶기)
taste_groups = {
    "맑다": ['맑다', '맑은'],
    "야들야들하다" : ['야들야들', '야들야들하다'],
    '쫄깃' : ['쫄깃', '쫄깃쫄깃'],
    '짜다' : ['짜다', '짭잘하다', '짭짤하다'],
    '맵다' : ['맵다', '매콤']
}

not_words = []

# 1. 그룹화 키워드 추출
group_keywords = set()
for v in taste_groups.values():
    group_keywords.update(v)
group_names = list(taste_groups.keys())

# 2. 그룹 미포함 키워드만 추출
individual_keywords = [kw for kw in taste_keywords if kw not in group_keywords]

for word in group_names + individual_keywords:
    not_words.append('NOT_'+word)

In [ ]:
total_keywords = individual_keywords + group_names + not_words

# 3. 컬럼명 만들기
keyword_columns = [f"{kw}_비율" for kw in total_keywords]
base_columns = ['가게명', '리뷰수','키워드포함_리뷰수',  '평균평점', '맛있다_수', '맛있다_비율']
total_info_df = pd.DataFrame(columns=base_columns + keyword_columns)

# 4. 계산 루프
for temp in [df_dggb_1, df_dggb_2, df_dggb_3, df_dggb_4, df_dggb_5, df_dggb_6]:
    review_count = len(temp)        
    temp = temp[temp['token'].apply(lambda tokens: any(token in tokens for token in total_keywords))]
    token_freq = Counter(chain.from_iterable(temp['token']))
    df_freq = pd.DataFrame(token_freq.items(), columns=['토큰', '빈도수'])


    review_count_keyword = len(temp)
    try:
        option = temp['세부옵션'].value_counts().idxmax()
    except:
        option = np.nan

    mean_score = temp['별점'].mean()
    delicious_count = df_freq[df_freq['토큰'] == '맛있다'].values[0][1] if '맛있다' in df_freq['토큰'].values else 0
    delicious_ratio = delicious_count / review_count_keyword if review_count_keyword > 0 else 0

    # 4-1. 그룹화 안 된 키워드 비율
    keyword_ratios = []
    for kw in individual_keywords:
        kw_count = temp['token'].apply(lambda tokens: kw in tokens).sum()
        kw_ratio = kw_count / review_count_keyword if review_count_keyword > 0 else 0
        keyword_ratios.append(kw_ratio)

    # 4-2. 그룹 비율
    for g in group_names:
        group_kw_list = taste_groups[g]
        group_count = temp['token'].apply(lambda tokens: any(kw in tokens for kw in group_kw_list)).sum()
        group_ratio = group_count / review_count_keyword if review_count_keyword > 0 else 0
        keyword_ratios.append(group_ratio)

    # 4-3. 부정어 비율
    for n in not_words:
        if n[4:] in individual_keywords:
            kw_count = temp['token'].apply(lambda tokens: n in tokens).sum()
            kw_ratio = kw_count / review_count_keyword if review_count_keyword > 0 else 0
            keyword_ratios.append(kw_ratio)
        else:
            group_kw_list = taste_groups[n[4:]]
            negated_kw_list = ['NOT_' + word for word in group_kw_list]
            group_count = temp['token'].apply(lambda tokens: any(kw in tokens for kw in negated_kw_list)).sum()
            group_ratio = group_count / review_count_keyword if review_count_keyword > 0 else 0
            keyword_ratios.append(group_ratio)


    total_info_df.loc[len(total_info_df)] = [np.nan, review_count, review_count_keyword, mean_score,
                                             delicious_count, delicious_ratio] + keyword_ratios



# 가게명 기입
total_info_df['가게명'] = ['건봉국밥', '꼰대국밥', '대건명가', '몽실종가', '수백당', '장사의신']
total_info_df = total_info_df.T
total_info_df.columns = total_info_df.iloc[0]
total_info_df = total_info_df[1:]  # 첫 번째 행 제거
total_info_df = total_info_df.loc[~(total_info_df == 0).all(axis=1)]

In [ ]:
pd.set_option('display.max_rows', None)       # 모든 행 출력
pd.set_option('display.max_columns', None)    # 모든 열 출력
total_info_df

In [ ]:
# 카테고리별 키워드 사전 정의
flavor_categories = {
    "맑고 깔끔한 맛": ["깔끔하다", "담백하다", "시원하다", "뽀얗다", "개운하다", "묽다", "맑다"],
    "양과 풍부함": ["넉넉하다", "든든하다", "실하다"],
    "진하고 깊은 맛": ["진하다", "구수하다", "깊다", "진국", "감칠맛", "걸쭉하다"],
    "고소": ["고소하다"],
    "매운맛/자극": ["얼큰하다", "칼칼하다", "자극", "맵다"],
    "느끼함/지방감": ["느끼하다"],
    "간 관련": ["싱겁다", "짜다"],
    "식감": ["퍽퍽", "야들야들하다", "쫄깃", "부드럽다"]
}


def label_flavor_category(index_word):
    for category, keywords in flavor_categories.items():
        for keyword in keywords:
            if keyword in index_word:  # 부분 일치 허용
                return category
    return "기타"

# df.index = df.index.astype(str)
total_info_df["범주"] = total_info_df.index.map(label_flavor_category)
total_info_df

In [ ]:
total_info_df['범주'].unique()

In [ ]:
total_info_df[total_info_df['범주'] == '식감']

In [ ]:
total_info_df.to_csv('./data/맛_키워드_비율_부정어구분.csv')